# Chatbot vs human resolution times: Kolmogorov-Smirnov test

**Two-Sample Kolmogorov-Smirnov Test** · a worked example from the StatNex teaching guide

**The story.** An online shop answers support tickets two ways: a chatbot (18 tickets) and a human agent (9 tickets). It recorded the minutes needed to resolve each ticket. Instead of asking which group is faster, we ask whether the whole pattern of resolution times is different.

**The question.** Do chatbot and human tickets follow different distributions of resolution time?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/ks_two_sample.csv` (18 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Chatbot,Human agent
5.0,14.0
30.0,12.0
4.0,16.0
7.0,13.0
6.0,19.0
42.0,11.0
3.0,15.0
5.0,17.0
9.0,13.0
6.0,
55.0,
4.0,
8.0,
7.0,
6.0,
5.0,
9.0,
70.0,

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
samples1 = ["Chatbot"]
samples2 = ["Human agent"]
# Each column once, in the order the pairs below use them.
columns = list(dict.fromkeys(name for a in samples1 for b in samples2 for name in (a, b)))
# One column per sample; text and blank cells are skipped, so the samples may differ
# in size.
values = {name: pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)
          for name in columns}
sizes = {name: len(values[name]) for name in columns}
print(pd.Series(sizes, name="n"))

## Two-sample Kolmogorov-Smirnov test

In [ ]:
# D is the largest gap between the two cumulative distributions: at some value, the
# share of sample 1 at or below it minus the share of sample 2. The P value is read
# from the limiting (large-sample) Kolmogorov distribution of
# KS = sqrt(n1 n2 / (n1 + n2)) D, scipy's stats.kstwobign. ks_2samp's own P (exact,
# or method="asymp") uses finite-sample forms and differs; in small samples the
# limiting P is conservative. It also ignores ties, so with heavily tied or grouped
# data treat P as approximate.
tests = []
for a in samples1:
    for b in samples2:
        n1, n2 = sizes[a], sizes[b]
        d = stats.ks_2samp(values[a], values[b], method="asymp").statistic
        ks = d * np.sqrt(n1 * n2 / (n1 + n2))
        p = stats.kstwobign.sf(ks)
        tests.append([a, b, d, ks, p, stars(p)])

tests = pd.DataFrame(tests, columns=["sample 1", "sample 2", "D", "KS", "P", ""])
print()
print(tests.to_string(index=False))

## What the result says

The two-sample Kolmogorov-Smirnov test compares two whole distributions, that is, the full patterns of the values, not only their centres. For every time, it compares the share of chatbot tickets already solved with the share of human agent tickets already solved.

The maximum difference between these two shares is 0.78: at one point most chatbot tickets are done while no human ticket is yet. From it comes the statistic KS = 1.91 and a two-sided P value of 0.0014. The P value is the chance of a gap this large if both channels had the same pattern. With 18 chatbot and 9 human agent tickets, that chance is very small.

The two channels have clearly different patterns of resolution time. But the test cannot say what differs (centre, spread or shape), or which channel is better.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.